# 07 · Preprocessing

**Objective:** Build the reusable, leakage-safe preprocessing pipeline that every model in notebooks 09–11 is built on, matching `src/ml/data_prep.py` plus the split/pipeline logic embedded in `src/ml/funding_regression.py` and `src/ml/success_classification.py`.

**Why this gets its own notebook, separate from feature engineering:** feature engineering (notebook 06) defines *what* columns exist. Preprocessing decides *how* they're split, imputed, encoded, and scaled for a specific model — and importantly, *when* those decisions are learned (train-only) versus applied (train + test), which is where data leakage bugs live.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

pd.set_option("display.max_columns", 30)

WH = Path("data/warehouse")
fact = pd.read_csv(WH / "fact_startup_funding.csv", low_memory=False)
startup = pd.read_csv(WH / "dim_startup_with_source_key.csv", low_memory=False)
geo = pd.read_csv(WH / "dim_geography.csv", low_memory=False)
industry = pd.read_csv(WH / "dim_industry.csv", low_memory=False)

df = fact.merge(startup, on="startup_id").merge(geo, on="geography_id", how="left").merge(industry, on="industry_id", how="left")

top_industries = df["primary_category"].value_counts().head(15).index
df["industry_grouped"] = np.where(df["primary_category"].isin(top_industries), df["primary_category"], "Other")
top_countries = df["country_name"].value_counts().head(15).index
df["country_grouped"] = np.where(df["country_name"].isin(top_countries), df["country_name"], "Other")
df["is_multi_round"] = df["is_multi_round"].astype(int)
df["has_debt_financing"] = df["has_debt_financing"].astype(int)

FEATURE_COLS_NUMERIC = ["funding_rounds", "years_since_founded", "is_multi_round",
                         "num_round_types_used", "has_debt_financing", "category_count"]
FEATURE_COLS_CATEGORICAL = ["industry_grouped", "country_grouped"]
print(f"Modeling table assembled: {df.shape[0]:,} rows")

Modeling table assembled: 67,098 rows


## 1. Data leakage — the one that matters most in this project

**The specific leakage risk here:** for the *success classification* use case (predict `is_exited`), using `funding_total_usd` directly as an input feature would be a subtle leakage bug. Not because it looks ahead in time in an obvious way, but because a startup that has *already* exited often continues to accumulate reported valuation/funding data points in the source (e.g. late secondary-market activity) that are downstream consequences of, or contemporaneous with, being exit-track — not knowledge a VC would have had *before* the outcome. I keep the engineered funding-behavior features (rounds, multi-round flag, round-type diversity) because they represent decisions made *during* the fundraising process, which a real analyst would have visibility into.

In [1]:
model_df_check = df.dropna(subset=["is_exited"]).copy()
corr_leak = model_df_check[["funding_total_usd", "is_exited"]].dropna()
print(f"Correlation of raw funding_total_usd with is_exited: {corr_leak['funding_total_usd'].corr(corr_leak['is_exited'].astype(int)):.3f}")
print(f"Correlation of funding_rounds (engineered, kept) with is_exited: "
      f"{model_df_check[['funding_rounds','is_exited']].dropna().corr().iloc[0,1]:.3f}")
print("\n-> funding_total_usd is EXCLUDED as a raw classification feature (see success_classification.py comment);")
print("   funding_rounds / is_multi_round / num_round_types_used ARE kept as legitimate early-signal features.")

Correlation of raw funding_total_usd with is_exited: 0.069
Correlation of funding_rounds (engineered, kept) with is_exited: 0.140

-> funding_total_usd is EXCLUDED as a raw classification feature (see success_classification.py comment);
   funding_rounds / is_multi_round / num_round_types_used ARE kept as legitimate early-signal features.


### Business interpretation
This distinction matters for how the classification model's predictions get used: if a VC wants to use this model on a startup *today*, they'd have visibility into how many rounds it has raised and what instrument types it has used — but not a "final" funding total that partially reflects outcomes still unfolding. Including the excluded feature wouldn't just be technically improper — it would make the model look artificially better in testing while being useless for real forward-looking decisions on an in-progress startup.

## 2. Train/test split strategy

In [1]:
# Regression use case: simple random split (target is continuous, no imbalance concern)
reg_df = df.dropna(subset=["log_funding_total_usd"] + FEATURE_COLS_NUMERIC + FEATURE_COLS_CATEGORICAL)
X_reg = reg_df[FEATURE_COLS_NUMERIC + FEATURE_COLS_CATEGORICAL]
y_reg = reg_df["log_funding_total_usd"]
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
print(f"Regression split -- train: {len(X_reg_train):,} | test: {len(X_reg_test):,}")

# Classification use case: STRATIFIED split, because the positive class (~11%) is imbalanced.
# A plain random split can, by chance, shift the test set's positive rate enough to make
# metrics noisy/incomparable across runs -- stratifying removes that noise source.
clf_df = df.dropna(subset=["is_exited"] + FEATURE_COLS_NUMERIC + FEATURE_COLS_CATEGORICAL)
X_clf = clf_df[FEATURE_COLS_NUMERIC + FEATURE_COLS_CATEGORICAL]
y_clf = clf_df["is_exited"].astype(int)
X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)
print(f"\nClassification split -- train: {len(X_clf_train):,} | test: {len(X_clf_test):,}")
print(f"  Overall positive rate: {y_clf.mean()*100:.2f}%")
print(f"  Train positive rate:   {y_clf_train.mean()*100:.2f}%")
print(f"  Test positive rate:    {y_clf_test.mean()*100:.2f}%")

Regression split -- train: 31,312 | test: 7,828

Classification split -- train: 37,647 | test: 9,412
  Overall positive rate: 11.25%
  Train positive rate:   11.25%
  Test positive rate:    11.25%


In [1]:
# Prove stratification matters: compare against what an unstratified split would have given
_, _, _, y_clf_test_unstratified = train_test_split(X_clf, y_clf, test_size=0.2, random_state=7)
print(f"Example UNSTRATIFIED split (different seed) test positive rate: {y_clf_test_unstratified.mean()*100:.2f}%")
print(f"Stratified split test positive rate (this project's approach): {y_clf_test.mean()*100:.2f}%")
print("-> stratification keeps the test set's class balance anchored to the true overall rate regardless of seed,")
print("   which is what makes ROC-AUC/precision/recall comparable across different train/test splits and reruns.")

Example UNSTRATIFIED split (different seed) test positive rate: 10.98%
Stratified split test positive rate (this project's approach): 11.25%
-> stratification keeps the test set's class balance anchored to the true overall rate regardless of seed,
   which is what makes ROC-AUC/precision/recall comparable across different train/test splits and reruns.


## 3. Reusable `ColumnTransformer` pipeline

I fit the pipeline **only on the training set**. This is the single most important leakage-prevention rule in this notebook: if the one-hot encoder or a scaler were fit on the full dataset (train+test) before splitting, the "unseen" test set would have silently leaked its category vocabulary / scale statistics into training — an easy, common, and hard-to-notice mistake.

In [1]:
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), FEATURE_COLS_CATEGORICAL),
], remainder="passthrough")

pipeline_demo = Pipeline([("prep", preprocessor)])

# Fit ONLY on training data
pipeline_demo.fit(X_reg_train)
X_train_transformed = pipeline_demo.transform(X_reg_train)
X_test_transformed = pipeline_demo.transform(X_reg_test)

print(f"Transformed training shape: {X_train_transformed.shape}")
print(f"Transformed test shape:     {X_test_transformed.shape}")
print(f"\nOneHotEncoder(handle_unknown='ignore') means a category seen only in the test set")
print(f"(not present in training) is encoded as all-zeros rather than raising an error --")
print(f"safe behavior for production scoring on unseen categories.")

Transformed training shape: (31312, 38)
Transformed test shape:     (7828, 38)

OneHotEncoder(handle_unknown='ignore') means a category seen only in the test set
(not present in training) is encoded as all-zeros rather than raising an error --
safe behavior for production scoring on unseen categories.


### Observation
Notice the transformer is fit once (`.fit(X_reg_train)`) and then applied to both sets (`.transform(...)`) — never re-fit on test data. `handle_unknown="ignore"` is a deliberate defensive choice: if a future startup belongs to an industry that didn't appear in training data, the pipeline degrades gracefully (all-zero encoding) instead of crashing in production.

## 4. Imputation strategy — why none is needed here

A textbook preprocessing pipeline usually needs a `SimpleImputer` step. Here, feature engineering (notebook 06) already resolved missingness at the source:

In [1]:
print("Remaining nulls in the final feature set (after notebook 06's engineering):")
print(reg_df[FEATURE_COLS_NUMERIC + FEATURE_COLS_CATEGORICAL].isna().sum())
print("\n-> zero remaining nulls: rows with missing target/features were already excluded via .dropna() above,")
print("   and structural zeros (round-type columns) were filled with 0 back in notebook 04/06, not left as NaN.")
print("   An imputer is demonstrated below for completeness / robustness against future schema drift.")

demo_pipeline = Pipeline([
    ("prep", ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), FEATURE_COLS_NUMERIC),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), FEATURE_COLS_CATEGORICAL),
    ])),
])
demo_pipeline.fit(X_reg_train)
print("\nDefensive pipeline (with imputers) fit successfully -- would tolerate future nulls without crashing.")

Remaining nulls in the final feature set (after notebook 06's engineering):
funding_rounds          0
years_since_founded     0
is_multi_round          0
num_round_types_used    0
has_debt_financing      0
category_count          0
industry_grouped        0
country_grouped         0
dtype: int64

-> zero remaining nulls: rows with missing target/features were already excluded via .dropna() above,
   and structural zeros (round-type columns) were filled with 0 back in notebook 04/06, not left as NaN.
   An imputer is demonstrated below for completeness / robustness against future schema drift.

Defensive pipeline (with imputers) fit successfully -- would tolerate future nulls without crashing.


## 5. Scaling — why tree-based models in this project skip it (and where it *would* matter)

Random Forest / Gradient Boosted Trees split on raw feature values and are invariant to monotonic rescaling — scaling `years_since_founded` from years to days wouldn't change a single tree's splits. Scaling *does* matter for the linear baseline model (notebook 09) and I apply it in the clustering use case (notebook 12, `StandardScaler` before K-Means, since K-Means uses Euclidean distance, which is scale-sensitive).

In [1]:
scaler = StandardScaler()
numeric_scaled = scaler.fit_transform(X_reg_train[FEATURE_COLS_NUMERIC])
print("Before scaling (training set numeric features):")
print(X_reg_train[FEATURE_COLS_NUMERIC].describe().loc[["mean", "std"]].round(2))
print("\nAfter StandardScaler (mean~0, std~1 by construction):")
print(pd.DataFrame(numeric_scaled, columns=FEATURE_COLS_NUMERIC).describe().loc[["mean", "std"]].round(2))

Before scaling (training set numeric features):
      funding_rounds  years_since_founded  is_multi_round  \
mean            1.83                 7.31            0.42   
std             1.37                10.44            0.49   

      num_round_types_used  has_debt_financing  category_count  
mean                  1.41                0.08            2.44  
std                   1.24                0.28            1.95  

After StandardScaler (mean~0, std~1 by construction):
      funding_rounds  years_since_founded  is_multi_round  \
mean            -0.0                  0.0             0.0   
std              1.0                  1.0             1.0   

      num_round_types_used  has_debt_financing  category_count  
mean                   0.0                -0.0            -0.0  
std                    1.0                 1.0             1.0  


### Observation
Post-scaling, every numeric feature has mean ~0 and std ~1 — confirming the scaler worked as intended. I skip this step for the tree-based models used as this project's primary predictors, but show it here because a complete preprocessing notebook should demonstrate it, and it *is* used for K-Means clustering (notebook 12), where the un-scaled `funding_total_usd` (in the billions) would otherwise completely dominate the Euclidean distance calculation over smaller-scale features like `category_count`.

## 6. Validation strategy: cross-validation instead of a separate fixed validation split

Rather than carving out a third fixed validation set (train/val/test), I use k-fold cross-validation *within* the training set for model comparison and hyperparameter tuning (notebooks 09–10), reserving the test set purely for a final, untouched evaluation. This uses the available data more efficiently than a fixed 3-way split, which matters given no dataset here is enormous once filtered to complete cases.

## 7. Summary — the reusable pipeline contract

| Decision | Choice | Why |
|---|---|---|
| Split type (regression) | random 80/20 | continuous target, no imbalance |
| Split type (classification) | stratified 80/20 | ~11% positive class — protects metric stability |
| Leakage-risk feature | `funding_total_usd` excluded from classification | outcome-adjacent, not available pre-decision |
| Encoding | `OneHotEncoder(handle_unknown="ignore")`, fit train-only | safe on unseen categories |
| Imputation | none needed (resolved in feature engineering); defensive imputer demonstrated | robustness |
| Scaling | none for tree models; `StandardScaler` for linear baseline & K-Means | distance/scale sensitivity differs by algorithm |
| Validation | k-fold CV within training set, not a fixed val split | efficient use of limited complete-case data |

## Interview questions this notebook prepares me for

- *"What's the most common preprocessing mistake you watch for?"* — Fitting any transformer (scaler, encoder, imputer) on the full dataset before splitting. It looks identical to correct code and usually doesn't error — it just quietly inflates test-set performance.
- *"Why stratify a train/test split?"* — To keep the target's class balance consistent between splits, so evaluation metrics on an imbalanced target are comparable across different random seeds/reruns.

## Next notebook
`08_Statistical_Analysis.ipynb` — confidence intervals, hypothesis tests, and a regression model with full assumption diagnostics on this same feature set.